# Práctica 2 — Clasificación multiclase con el dataset Wine

**Objetivo de esta práctica:** el mismo tipo de problema que con Iris (clasificación multiclase), pero con un dataset un poco más grande: 13 características químicas por muestra de vino, en vez de 4. Sirve para confirmar que el mismo patrón (escalar → red densa → softmax → EarlyStopping) funciona igual sin importar cuántas columnas tenga el dataset.

Al final se agregan dos bloques extra para comparar arquitecturas de salida: una para clasificación binaria (`sigmoid`) y otra para regresión (`linear`), y entender por qué cada una usa una función de pérdida distinta.

## 1. Importar librerías

In [1]:
import tensorflow as tf                                            # TensorFlow: para construir y entrenar la red neuronal
import matplotlib.pyplot as plt                                    # Para graficar (no se usa en esta práctica, pero queda listo)
from sklearn.datasets import load_wine                             # Carga el dataset Wine ya incluido en scikit-learn
from sklearn.model_selection import train_test_split               # Divide los datos en entrenamiento y prueba
from sklearn.preprocessing import StandardScaler                   # Normaliza los datos (misma escala para todas las columnas)
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score   # Para evaluar qué tan bien predijo el modelo


## 2. Cargar el dataset y separar entrenamiento / prueba

`load_wine()` trae 178 muestras de vino, cada una con 13 medidas químicas (`x`, debe ser una matriz 2D) y su tipo de vino ya etiquetado (`y`, debe ser un vector 1D con valores 0, 1 o 2).

In [2]:
vino = load_wine()          # Carga el dataset completo (datos + etiquetas + nombres)
x = vino.data               # x: 13 características químicas por muestra (matriz 178 filas x 13 columnas) -> debe ser 2D
y = vino.target             # y: el tipo de vino de cada muestra, como número (0, 1 o 2) -> debe ser 1D

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42        # 20% para prueba, 80% para entrenar; random_state fija la mezcla para que sea repetible
)


## 3. Escalar los datos (`StandardScaler`)

Igual que en Iris: se ajusta el escalador SOLO con el train y se aplica esa misma transformación al test, para no filtrar información de los datos de prueba dentro del entrenamiento.

In [3]:
scaler = StandardScaler()                        # Crea el objeto que va a normalizar los datos
x_train = scaler.fit_transform(x_train)          # Calcula la media/desviación con el train y transforma el train
x_test = scaler.transform(x_test)                # Usa esa MISMA media/desviación para transformar el test


## 4. Construir el modelo

Arquitectura: 13 entradas → capa oculta de 40 neuronas → capa oculta de 20 neuronas → salida de 3 neuronas con `softmax` (3 tipos de vino posibles). Al tener más columnas de entrada que Iris, se usan capas ocultas un poco más grandes.

In [4]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(40, activation='relu', input_shape=(13,)),   # 1ra capa oculta: 40 neuronas; input_shape=(13,) = 13 entradas
    tf.keras.layers.Dense(20, activation='relu'),                      # 2da capa oculta: 20 neuronas
    tf.keras.layers.Dense(3, activation='softmax')                     # Capa de salida: 3 neuronas (una por tipo de vino), softmax
])


c:\Users\ANT DOR\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## 5. Compilar el modelo

Mismo esquema que Iris: `sparse_categorical_crossentropy` porque las etiquetas son enteros, no vectores one-hot.

In [5]:
model.compile(
    optimizer="adam",                              # Optimizador: ajusta los pesos en cada paso de entrenamiento
    loss="sparse_categorical_crossentropy",        # Pérdida correcta para clasificación multiclase con etiquetas enteras
    metrics=["accuracy"]                           # Queremos ver el % de aciertos durante el entrenamiento
)


## 6. `EarlyStopping`

Igual que en Iris, pero aquí se agrega `min_delta`: exige que la mejora sea de al menos 0.001 para contarla como "mejora real" — así no sigue entrenando por cambios microscópicos en el loss.

In [6]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="loss",                 # Qué métrica vigila para decidir si detener el entrenamiento
    patience=5,                      # Cuántas épocas espera sin mejora antes de detenerse
    restore_best_weights=True,       # Al terminar, se queda con los pesos de la mejor época
    min_delta=0.001                  # Umbral mínimo de mejora para que cuente como progreso real
)


## 7. Entrenar el modelo

In [7]:
model.fit(
    x_train, y_train,
    epochs=500,                      # Número máximo de épocas
    callbacks=[early_stop]           # EarlyStopping decide cuándo detenerse antes de llegar a 500
)


Epoch 1/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.2817 - loss: 1.1928
Epoch 2/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3028 - loss: 1.0911 
Epoch 3/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3521 - loss: 1.0075
Epoch 4/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4366 - loss: 0.9363  
Epoch 5/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5070 - loss: 0.8771
Epoch 6/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5845 - loss: 0.8211 
Epoch 7/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6549 - loss: 0.7681
Epoch 8/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6972 - loss: 0.7191
Epoch 9/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7394 - loss: 0.6736
Epoch 10/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7746 - loss: 0.6273  
Epoch 11/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8380 - loss: 0.5834
Epoch 12/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - accuracy: 0.8803 -

## 8. Predecir con los datos de prueba

In [8]:
predicciones = model.predict(x_test).argmax(1)      # Para cada muestra de prueba, elige el tipo de vino más probable
print("Predicciones del modelo")
print(predicciones)
print("Etiquetas reales de los datos de prueba")
print(y_test)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicciones del modelo
[0 0 2 0 1 0 1 2 1 2 0 2 0 1 0 1 1 1 0 1 0 1 1 2 2 2 1 1 1 0 0 1 2 0 0 0]
Etiquetas reales de los datos de prueba
[0 0 2 0 1 0 1 2 1 2 0 2 0 1 0 1 1 1 0 1 0 1 1 2 2 2 1 1 1 0 0 1 2 0 0 0]


## 9. Evaluar qué tan bien predijo el modelo

In [9]:
print(accuracy_score(y_test, predicciones))              # % de predicciones correctas sobre el total
print(classification_report(y_test, predicciones))       # precision, recall y f1-score por cada tipo de vino
print(confusion_matrix(y_test, predicciones))             # matriz de confusión: aciertos y errores por clase


1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00         8

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

[[14  0  0]
 [ 0 14  0]
 [ 0  0  8]]


## 10. Resumen de métricas (mismo resultado, presentado más ordenado)

In [10]:
print("--- Métricas de evaluación del modelo ---")
print(classification_report(y_test, predicciones))


--- Métricas de evaluación del modelo ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00         8

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [11]:
print("--- Métricas de evaluación del modelo ---")
print("Matriz de confusión")
print(confusion_matrix(y_test, predicciones))
print("Precisión del modelo:", accuracy_score(y_test, predicciones))


--- Métricas de evaluación del modelo ---
Matriz de confusión
[[14  0  0]
 [ 0 14  0]
 [ 0  0  8]]
Precisión del modelo: 1.0


## 11. Comparación de arquitecturas de salida (bloque extra)

Estos dos modelos NO se entrenan (no tienen datos ni `.fit`); son solo para comparar cómo cambia la ÚLTIMA capa y la función de pérdida según el tipo de problema. Es la tabla de referencia que hay que memorizar:

| Tipo de problema | Activación de salida | Función de pérdida |
|---|---|---|
| Clasificación binaria (2 clases) | `sigmoid` | `binary_crossentropy` |
| Regresión (predecir un número) | `linear` | `mse` |
| Clasificación multiclase (3+ clases) | `softmax` | `sparse_categorical_crossentropy` |

### 11.1 Ejemplo: clasificación binaria (`sigmoid` + `binary_crossentropy`)

In [12]:
import tensorflow as tf

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(2,)),                 # Ejemplo con 2 entradas cualquiera
    tf.keras.layers.Dense(4, activation='relu'),        # Capa oculta normal
    tf.keras.layers.Dense(1, activation='sigmoid'),     # 1 sola neurona de salida: sigmoid da un valor entre 0 y 1 (probabilidad de la clase 1)
])
model.compile(optimizer='adam', loss='binary_crossentropy')   # Pérdida correcta cuando la salida es sigmoid (2 clases)


### 11.2 Ejemplo: regresión (`linear` + `mse`)

In [13]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(2,)),                 # Ejemplo con 2 entradas cualquiera
    tf.keras.layers.Dense(4, activation='relu'),        # Capa oculta normal
    tf.keras.layers.Dense(1, activation='linear'),      # 1 sola neurona de salida: linear = predice un número cualquiera (no una probabilidad)
])
model.compile(optimizer='sgd', loss='mse')             # mse (error cuadrático medio) es la pérdida estándar para regresión


## 12. Conclusión de esta práctica

El modelo de Wine también llegó a accuracy = 1.0, confirmando que la misma arquitectura básica (escalar → Dense → softmax → EarlyStopping) funciona con datasets de distinto tamaño. La parte más importante para recordar de esta práctica es la tabla de arquitecturas de salida: elegir mal la combinación activación/pérdida (por ejemplo, usar `softmax` con `mse`) es de los errores más comunes al armar una red nueva.